# Housing HPI forecast

Can we use this quarter’s data to predict next quarter’s house price index (HPI)? The index tracks changes in home prices. We measure it in points.

**Run the notebook.** Select the first cell and press **Shift + Enter**. Repeat one cell at a time, from top to bottom. The two output folders start empty. Each time a cell saves a file, it shows the file name and folder below it.

## 1. Open the data

The Excel file has one row per market and quarter. A quarter is three months. The data cover the 50 US states and Washington, DC, from 2002 to 2016.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

data_file = Path("Dataset") / "Modeled_Dataset_2002_2016.xlsx"
tables_dir = Path("Loader") / "Excel_Run"
figures_dir = Path("Loader") / "Figures"
for folder in (tables_dir, figures_dir):
    folder.mkdir(parents=True, exist_ok=True)
    (folder / ".gitkeep").unlink(missing_ok=True)

def save_sheet(table, name):
    table.to_csv(tables_dir / name, index=False, float_format="%.8f")
    print(f"Sheet {name} has loaded successfully inside {tables_dir.as_posix()}")

def show_and_save_chart(fig, name):
    fig.savefig(figures_dir / name, dpi=180, facecolor="white")
    plt.show()
    print(f"Image {name} has loaded successfully inside {figures_dir.as_posix()}")

navy, teal = "#263D60", "#087C76"
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 10, "axes.titleweight": "bold"})

In [ ]:
panel = pd.read_excel(data_file, sheet_name="ML_State_Quarter")
panel = panel.sort_values(["PERIOD", "State"]).reset_index(drop=True)
panel["TargetPeriod"] = (pd.PeriodIndex(panel["PERIOD"], freq="Q") + 1).astype(str)
print(f"Read {len(panel):,} rows from the ML_State_Quarter sheet.")
display(panel.head())

## 2. Choose the inputs and the target

| Role | Column | Meaning |
|---|---|---|
| Input | `State` | State or DC |
| Input | `HPI_SA` | This quarter’s HPI, adjusted for seasonal changes |
| Input | `Mortgage_90PlusDelinq_pct` | Share of mortgages at least 90 days late |
| Input | `ForeclosureRelated_pct` | Share of mortgages linked to foreclosure |
| Target | `TARGET_NextQ_HPI_SA` | The same market’s HPI next quarter |

The two mortgage shares are percentages. Foreclosure is when a lender takes back a home after missed payments.

In [ ]:
target = "TARGET_NextQ_HPI_SA"
numeric_inputs = ["HPI_SA", "Mortgage_90PlusDelinq_pct", "ForeclosureRelated_pct"]
inputs = ["State", *numeric_inputs]

# Match each target to the same state's HPI one quarter later.
history = panel.set_index(["State", "PERIOD"], verify_integrity=True)["HPI_SA"]
next_quarters = pd.MultiIndex.from_frame(panel[["State", "TargetPeriod"]])
expected_target = history.reindex(next_quarters).reset_index(drop=True)
if panel[inputs].isna().any().any() or not panel[target].equals(expected_target):
    raise ValueError("Check the inputs and next-quarter HPI values in the workbook.")

## 3. Look at Nevada’s housing market

The line shows how Nevada’s HPI changed over time. The labels mark the peak and the lowest point after it.

In [ ]:
nv_history = panel.loc[panel["State"].eq("NV")].set_index("PERIOD")["HPI_SA"]
peak = nv_history.idxmax()
low = nv_history.loc[peak:].idxmin()

fig, ax = plt.subplots(figsize=(9, 3.8), layout="constrained")
ax.plot(nv_history.index, nv_history, color=teal, linewidth=2.5)
for quarter, label, offset in [(peak, "Peak", 11), (low, "Low after peak", -17)]:
    value = nv_history.loc[quarter]
    ax.scatter(quarter, value, color=navy, zorder=3)
    ax.annotate(f"{label} · {quarter} · {value:.2f}", (quarter, value),
                xytext=(0, offset), textcoords="offset points", ha="center", color=navy)
years = nv_history.index[nv_history.index.str.endswith("Q1")]
ax.set_xticks(years, labels=years.str[:4])
ax.set(title="Nevada house price index, 2002–2016", xlabel="Year", ylabel="HPI points")
ax.margins(x=0.02, y=0.22)
show_and_save_chart(fig, "01_Nevada_HPI_History.png")

## 4. Split the data by time

We split by the quarter we want to predict. The model learns from the early quarters. We test it on later quarters, one quarter ahead at a time.

| Group | Quarter we predict | Use |
|---|---|---|
| Training | 2002 Q2 to 2012 Q4 | Teach the model |
| Test | 2013 Q1 to 2016 Q4 | Check predictions against known values |
| Forecast | 2017 Q1 | Predict a quarter with no known result in this file |

`ML_Training.csv` contains the original training rows and the quarter each row predicts.

In [ ]:
train = panel.loc[panel["TargetPeriod"].between("2002Q2", "2012Q4")].copy()
test = panel.loc[panel["TargetPeriod"].between("2013Q1", "2016Q4")].copy()
future = panel.loc[panel["TargetPeriod"].eq("2017Q1")].copy()

print(f"Training rows: {len(train):,} | Test rows: {len(test):,} | Forecast rows: {len(future):,}")
display(train.head())
save_sheet(train, "ML_Training.csv")

## 5. Train a model and make test predictions

We use **Linear Regression** because next quarter’s HPI is a number. The model learns how the inputs relate to HPI from the training rows. It turns each state code into a yes/no input.

The table compares the model’s prediction with the actual HPI for the same quarter.

In [ ]:
prepare_inputs = ColumnTransformer([
    ("state", OneHotEncoder(handle_unknown="ignore"), ["State"]),
    ("numbers", "passthrough", numeric_inputs),
])
model = make_pipeline(prepare_inputs, LinearRegression())
model.fit(train[inputs], train[target])

def prediction_table(rows, fitted_model):
    table = rows[["State", "PERIOD", "TargetPeriod", "HPI_SA", target]].rename(columns={
        "State": "StateCode", "PERIOD": "OriginPeriod",
        "HPI_SA": "HPI_Origin", target: "Actual_HPI_SA",
    })
    table["Pred_HPI_SA"] = fitted_model.predict(rows[inputs])
    return table

test_predictions = prediction_table(test, model)
display(test_predictions.head().round(2))
save_sheet(test_predictions, "ML_Test_Predictions.csv")

## 6. Check how well the model did

- **MAE** is the average size of a prediction error, in HPI points. Lower is better.
- **RMSE** also measures error in HPI points. It gives more weight to big errors. Lower is better.
- **R²** shows how well the model explains differences in HPI. A score of 1 means a perfect fit.

In [ ]:
def scores(group, actual, predicted):
    return {
        "Dataset": group,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
    }

metrics = pd.DataFrame([
    scores("Training", train[target], model.predict(train[inputs])),
    scores("Test", test_predictions["Actual_HPI_SA"], test_predictions["Pred_HPI_SA"]),
])
display(metrics.round(4))
test_mae = metrics.loc[metrics["Dataset"].eq("Test"), "MAE"].item()
print(f"On the test data, the predictions missed the actual HPI by {test_mae:.2f} points on average.")
save_sheet(metrics, "ML_Train_Test_Metrics.csv")

### Compare training and test scores

The left chart shows errors in HPI points. The right chart shows R² on its own scale.

In [ ]:
fig, (err_ax, fit_ax) = plt.subplots(1, 2, figsize=(9.6, 3.9), layout="constrained",
                                    gridspec_kw={"width_ratios": [1.3, 1]})
by_group = metrics.set_index("Dataset")
by_group[["MAE", "RMSE"]].T.plot.bar(ax=err_ax, color=[navy, teal], rot=0, width=0.68)
for bars in err_ax.containers:
    err_ax.bar_label(bars, fmt="%.2f", padding=3)
err_ax.set(title="Prediction error", xlabel="", ylabel="HPI points")
err_ax.set_ylim(0, by_group[["MAE", "RMSE"]].max().max() * 1.28)
err_ax.legend(title="", ncol=2, loc="upper left")

bars = fit_ax.bar(by_group.index, by_group["R2"], color=[navy, teal], width=0.55)
fit_ax.bar_label(bars, fmt="%.4f", padding=3)
fit_ax.set(title="Model fit", ylabel="R² score", ylim=(0, 1.12), yticks=[0, 0.25, 0.5, 0.75, 1])
for axis in (err_ax, fit_ax):
    axis.grid(axis="x", visible=False)
show_and_save_chart(fig, "04_Train_Test_Evaluation.png")

### Compare predictions with actual values

The line chart follows Nevada over time. The dot chart shows every test row. A dot on the dashed line means the prediction matched the actual HPI.

In [ ]:
nv_test = test_predictions.loc[test_predictions["StateCode"].eq("NV")].set_index("TargetPeriod")
fig, ax = plt.subplots(figsize=(9, 3.8), layout="constrained")
ax.plot(nv_test.index, nv_test["Actual_HPI_SA"], label="Actual", color=navy, linewidth=2.4)
ax.plot(nv_test.index, nv_test["Pred_HPI_SA"], label="Predicted", color=teal,
        linestyle="--", linewidth=2)
ax.set_xticks(nv_test.index[::2])
ax.set(title="Nevada's actual and predicted HPI", xlabel="Quarter predicted", ylabel="HPI points")
ax.legend()
show_and_save_chart(fig, "02_Nevada_Test_Predictions.png")

In [ ]:
actual = test_predictions["Actual_HPI_SA"]
predicted = test_predictions["Pred_HPI_SA"]
limits = [min(actual.min(), predicted.min()), max(actual.max(), predicted.max())]

fig, ax = plt.subplots(figsize=(6.2, 4.6), layout="constrained")
ax.scatter(actual, predicted, s=21, alpha=0.4, color=teal, edgecolors="none")
ax.plot(limits, limits, "--", color=navy, label="Exact match")
ax.set(title=f"Actual and predicted HPI · {len(actual)} test rows",
       xlabel="Actual HPI (points)", ylabel="Predicted HPI (points)", aspect="equal")
ax.legend()
show_and_save_chart(fig, "03_Actual_vs_Predicted.png")

## 7. Predict the first quarter of 2017

Now we train a fresh copy of the model on all rows with a known target. We use the last quarter of 2016 to predict the next quarter for each market.

`ML_Predictions.csv` holds the test predictions and the new forecasts. The actual HPI is blank for 2017 Q1 because the data end in 2016.

In [ ]:
known = panel.loc[panel[target].notna()]
forecast_model = clone(model)
forecast_model.fit(known[inputs], known[target])
forecast = prediction_table(future, forecast_model)

all_predictions = pd.concat([
    test_predictions.assign(RowType="Test"),
    forecast.assign(RowType="Forecast"),
], ignore_index=True)
print(f"Forecasts for {len(forecast)} markets. The first five are shown below.")
display(forecast[["StateCode", "TargetPeriod", "HPI_Origin", "Pred_HPI_SA"]].head().round(2))
save_sheet(all_predictions, "ML_Predictions.csv")

### What does this mean for a housing team?

Nevada is one example. Compare the expected change with the average test error when using the forecast to plan a budget.

In [ ]:
nv = forecast.loc[forecast["StateCode"].eq("NV")].iloc[0]
change = nv["Pred_HPI_SA"] - nv["HPI_Origin"]
change_pct = 100 * change / nv["HPI_Origin"]
print(f"Nevada's HPI was {nv['HPI_Origin']:.2f} in 2016 Q4. The forecast for 2017 Q1 is {nv['Pred_HPI_SA']:.2f}.")
print(f"Expected change: {change:+.2f} HPI points ({change_pct:+.2f}%).")
print("A housing team can plan for a small rise and also budget for flat prices.")
print(f"The average test error across all markets was {test_mae:.2f} points, close to the size of this rise.")